# 29 — Quantization Deep Dive: PTQ, QAT, GPTQ, AWQ, GGUF and QLoRA

**Network LLM Engineer Certification — Model Efficiency**

### Learning goals
- Distinguish training-time and inference-time quantization strategies
- Choose a quantization path for Mac/CPU, single-GPU and server deployment
- Benchmark quality, memory and latency rather than selecting by file size alone

## The taxonomy

**Quantization** is not one method.

### Post-training quantization (PTQ)
Quantize an already-trained model without retraining the full model.

Examples/approaches:
- weight-only INT8 / INT4,
- **GPTQ**,
- **AWQ**,
- GGUF quantized model distributions used heavily with llama.cpp-style runtimes.

### Quantization-aware training (QAT)
Simulate quantization effects during training so the model can adapt to them.

### QLoRA
A training method:
- frozen quantized base weights (commonly 4-bit),
- train LoRA adapters in higher-precision compute.

These solve related but different problems.

In [ ]:
# Raw weight-memory intuition only; actual runtime memory is higher.
models = [0.6, 3, 8, 14, 32]
for b in models:
    print(f"\n{b}B parameters")
    for bits in [16, 8, 4]:
        gb = b * 1e9 * bits / 8 / 1e9
        print(f"  {bits:2d}-bit raw weights ~= {gb:5.1f} GB")

## GGUF

GGUF is primarily a **model file format/ecosystem** used by `llama.cpp` and compatible runtimes.
It packages model tensors and metadata and is commonly distributed in multiple quantization levels.

Use case:
- laptop/CPU/Apple Silicon experimentation,
- edge or local inference,
- partial GPU offload.

Do not confuse the file format with one universal quantization algorithm.

## GPTQ

GPTQ is a post-training weight quantization family intended to minimize quantization error using calibration data.

Think:
`trained model -> calibration samples -> quantization optimization -> quantized model`

The calibration set matters. A model quantized using irrelevant text may preserve the wrong behavior better than your networking workload.

## AWQ

Activation-aware Weight Quantization uses activation behavior from calibration data to identify important weights/channels and reduce degradation.

For Network AI, calibration should resemble production input:
- CLI,
- incident text,
- structured JSON,
- long runbooks,
not only generic prose.

## QAT

Quantization-aware training introduces/simulates quantization effects during training.
It can preserve accuracy better than naive PTQ in some scenarios but costs training complexity.

For most network-engineer labs, start with PTQ/QLoRA before escalating to QAT.

## Selection matrix

| Goal | Strong starting point |
|---|---|
| MacBook / CPU local assistant | GGUF + llama.cpp/Ollama-compatible runtime |
| NVIDIA GPU server | BF16/FP16 first, then benchmark AWQ/GPTQ/other supported kernels |
| Fine-tune larger model on limited VRAM | QLoRA / NF4 |
| Edge deployment | aggressive quantization + workload-specific benchmark |
| Highest correctness, enough VRAM | BF16/FP16 baseline |

The exact best method is hardware- and runtime-specific.

In [ ]:
# Tiny quantization-error demonstration.
import numpy as np
rng = np.random.default_rng(42)
weights = rng.normal(0, 0.7, size=10000)

def symmetric_quantize(x, bits):
    qmax = 2**(bits-1)-1
    scale = np.max(np.abs(x))/qmax
    q = np.clip(np.round(x/scale), -qmax, qmax)
    xhat = q*scale
    return xhat

for bits in [8,4,3]:
    xhat = symmetric_quantize(weights, bits)
    mse = np.mean((weights-xhat)**2)
    print(bits, "bit MSE:", round(float(mse), 6))

## Certification benchmark

For any quantized model candidate measure:

- held-out network correctness,
- schema validity,
- hallucinated-state rate,
- first-token latency,
- output throughput,
- peak RAM/VRAM,
- cold-start time,
- max practical context,
- tool-call correctness.

A 4-bit model that fits comfortably but loses critical BGP-policy accuracy may be a bad production trade.

### Exercise

Choose deployment for three cases:

1. 8B Network Assistant on a MacBook Pro.
2. 14B on one 24 GB NVIDIA GPU.
3. High-throughput on-prem API on multiple data-center GPUs.

For each, choose precision/quantization/runtime and define the benchmark that would falsify your choice.